# Kepler — Interactive Newtonian Orbit Laboratory

Explore the planar Newtonian two-body problem with editable initial conditions, saved scenarios, animated trajectories, diagnostics, and exportable reports. The implementation uses structured functions and a velocity-Verlet integrator.

> **GitHub note:** GitHub renders this notebook as a static document. Clone it and run it in JupyterLab to activate the controls and animation.

## Physical model

For bodies at $\mathbf r_1$ and $\mathbf r_2$, the accelerations are

$$\mathbf a_1 = Gm_2\frac{\mathbf r_2-\mathbf r_1}{|\mathbf r_2-\mathbf r_1|^3}, \qquad \mathbf a_2 = Gm_1\frac{\mathbf r_1-\mathbf r_2}{|\mathbf r_2-\mathbf r_1|^3}.$$

The default scenarios use normalized units with $G=1$. Any consistent units work if the matching value of $G$ is supplied. Reduce `dt` if the conservation-error plot grows too much, especially during close encounters.

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.orbit import (PRESETS, conservation_summary, diagnostics_figure,
                       export_results, get_preset, simulate,
                       to_dataframe, trajectory_figure)

print(f'Project root: {PROJECT_ROOT}')
print(f'Available scenarios: {len(PRESETS)}')

## Interactive laboratory

Choose a saved scenario first, then edit any value. **Load preset** restores the selected scenario, **Run simulation** calculates and displays it, and **Export results** saves the latest run under `exports/`.

In [ ]:
style = {'description_width': '115px'}
wide = widgets.Layout(width='260px')
short = widgets.Layout(width='235px')

preset = widgets.Dropdown(options=list(PRESETS), description='Scenario', style=style, layout=widgets.Layout(width='420px'))
description = widgets.HTML()
m1 = widgets.FloatText(description='Mass 1', style=style, layout=short)
m2 = widgets.FloatText(description='Mass 2', style=style, layout=short)
radius1 = widgets.FloatText(description='Radius 1', style=style, layout=short)
radius2 = widgets.FloatText(description='Radius 2', style=style, layout=short)
G = widgets.FloatText(description='G', style=style, layout=short)
dt = widgets.FloatText(description='Time step', style=style, layout=short)
duration = widgets.FloatText(description='Duration', style=style, layout=short)
r1x = widgets.FloatText(description='Body 1 x', style=style, layout=short)
r1y = widgets.FloatText(description='Body 1 y', style=style, layout=short)
r2x = widgets.FloatText(description='Body 2 x', style=style, layout=short)
r2y = widgets.FloatText(description='Body 2 y', style=style, layout=short)
v1x = widgets.FloatText(description='Body 1 vx', style=style, layout=short)
v1y = widgets.FloatText(description='Body 1 vy', style=style, layout=short)
v2x = widgets.FloatText(description='Body 2 vx', style=style, layout=short)
v2y = widgets.FloatText(description='Body 2 vy', style=style, layout=short)

load_button = widgets.Button(description='Load preset', icon='refresh', button_style='info')
run_button = widgets.Button(description='Run simulation', icon='play', button_style='success')
export_button = widgets.Button(description='Export results', icon='download', button_style='warning', disabled=True)
status = widgets.HTML()
output = widgets.Output()
latest = {'result': None}

def load_selected(_=None):
    config = get_preset(preset.value)
    m1.value, m2.value = config['m1'], config['m2']
    radius1.value, radius2.value = config['radius1'], config['radius2']
    G.value, dt.value, duration.value = config['G'], config['dt'], config['duration']
    r1x.value, r1y.value = config['r1']
    r2x.value, r2y.value = config['r2']
    v1x.value, v1y.value = config['v1']
    v2x.value, v2y.value = config['v2']
    description.value = f"<i>{config['description']}</i>"
    status.value = '<span style="color:#64748b">Preset loaded. Press Run simulation.</span>'

def current_config():
    return {'description': description.value, 'm1': m1.value, 'm2': m2.value,
            'radius1': radius1.value, 'radius2': radius2.value, 'G': G.value,
            'dt': dt.value, 'duration': duration.value,
            'r1': [r1x.value, r1y.value], 'r2': [r2x.value, r2y.value],
            'v1': [v1x.value, v1y.value], 'v2': [v2x.value, v2y.value]}

def run_simulation(_=None):
    run_button.disabled = True
    export_button.disabled = True
    status.value = '<b>Calculating…</b>'
    try:
        result = simulate(current_config())
        latest['result'] = result
        summary = conservation_summary(result)
        with output:
            clear_output(wait=True)
            display(HTML(
                f"<h3>{summary['classification']}</h3>"
                f"<b>Samples:</b> {summary['samples']:,} &nbsp; "
                f"<b>Final time:</b> {summary['simulated_time']:.6g} &nbsp; "
                f"<b>Minimum separation:</b> {summary['minimum_separation']:.6g}<br>"
                f"<b>Maximum relative energy error:</b> {summary['max_relative_energy_error']:.3e} &nbsp; "
                f"<b>Maximum relative angular momentum error:</b> {summary['max_relative_angular_momentum_error']:.3e}"))
            display(trajectory_figure(result, animated=True))
            display(diagnostics_figure(result))
            display(HTML('<h3>First and last data samples</h3>'))
            display(to_dataframe(result).iloc[[0, -1]])
        status.value = '<span style="color:#16a34a"><b>Simulation complete.</b></span>'
        export_button.disabled = False
    except Exception as error:
        status.value = f'<span style="color:#dc2626"><b>Error:</b> {error}</span>'
    finally:
        run_button.disabled = False

def export_simulation(_=None):
    if latest['result'] is None:
        return
    export_button.disabled = True
    status.value = '<b>Exporting…</b>'
    files = export_results(latest['result'], PROJECT_ROOT / 'exports')
    links = []
    notes = []
    for label, path in files.items():
        if label.endswith('_note'):
            notes.append(str(path))
        else:
            relative = Path(path).relative_to(PROJECT_ROOT)
            links.append(f'<li><code>{relative}</code></li>')
    status.value = '<span style="color:#16a34a"><b>Export complete:</b></span><ul>' + ''.join(links) + '</ul>' + '<br>'.join(notes)
    export_button.disabled = False

load_button.on_click(load_selected)
run_button.on_click(run_simulation)
export_button.on_click(export_simulation)
preset.observe(lambda change: load_selected() if change['name'] == 'value' else None, names='value')
load_selected()

initial_conditions = widgets.VBox([
    widgets.HTML('<b>Positions</b>'), widgets.HBox([r1x, r1y]), widgets.HBox([r2x, r2y]),
    widgets.HTML('<b>Velocities</b>'), widgets.HBox([v1x, v1y]), widgets.HBox([v2x, v2y])])
physical_settings = widgets.VBox([
    widgets.HTML('<b>Physical and numerical settings</b>'), widgets.HBox([m1, m2]),
    widgets.HBox([radius1, radius2]), widgets.HBox([G, dt]), duration])
controls = widgets.VBox([preset, description, widgets.HBox([initial_conditions, physical_settings]),
                         widgets.HBox([load_button, run_button, export_button]), status])
display(controls, output)

## Interpreting the results

- **Bound orbit:** negative relative specific orbital energy.
- **Escape / unbound:** positive energy and outward radial motion at the end.
- **Marginal escape:** relative specific energy is approximately zero.
- **Collision:** separation reached the sum of the configured radii.
- The conservation curves should remain close to zero. A large drift means the time step should be reduced.

The classification describes the simulated terminal state; it is not a general proof of long-term stability.